# Remote data access using pandas

The pandas library enables access to data displayed on websites using the `read_html()` function and access to the API endpoints of various data providers through the related `pandas-datareader` library.

In [7]:
import os
import pandas_datareader.data as web
from datetime import datetime
from pprint import pprint
import pandas as pd
import yfinance as yf

## Download html table with SP500 constituents

The download of the content of one or more html tables works as follows, for instance for the constituents of the S&P500 index from Wikipedia

In [3]:
sp_url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
sp500_constituents = pd.read_html(sp_url, header=0)[0]

In [4]:
sp500_constituents.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Symbol                 503 non-null    object
 1   Security               503 non-null    object
 2   GICS Sector            503 non-null    object
 3   GICS Sub-Industry      503 non-null    object
 4   Headquarters Location  503 non-null    object
 5   Date added             503 non-null    object
 6   CIK                    503 non-null    int64 
 7   Founded                503 non-null    object
dtypes: int64(1), object(7)
memory usage: 31.6+ KB


In [5]:
sp500_constituents.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


## pandas-datareader for Market Data

`pandas` used to facilitate access to data providers' APIs directly, but this functionality has moved to the related pandas-datareader library. The stability of the APIs varies with provider policies, and as of June 2o18 at version 0.7, the following sources are available

See [documentation](https://pandas-datareader.readthedocs.io/en/latest/); functionality frequently changes as underlying provider APIs evolve.

### Yahoo Finance

In [10]:
start = '2023-01-01'
end = datetime(2025, 5, 24)

# Download data
df = yf.download('META', start=start, end=end)

# Check the data
print(df.head())


[*********************100%***********************]  1 of 1 completed

Price            Close        High         Low        Open    Volume
Ticker            META        META        META        META      META
Date                                                                
2023-01-03  124.059395  125.680506  121.612818  122.149873  35528500
2023-01-04  126.675056  128.345890  125.163346  126.684996  32397100
2023-01-05  126.247398  127.818779  123.860492  125.441813  25447100
2023-01-06  129.310593  129.618899  125.352305  128.266319  27584500
2023-01-09  128.763596  132.224604  128.574630  130.444377  26649100


### Quandl

Need API key. To get API key and set it as an environment variable,follow the instructions below:

1. go to https://data.nasdaq.com/sign-up
2. Sign up and you will get an API key displayed in the page after your scucesfully signed up.
3. Permanatantly add the API key into your environment varialbe.
The method varies depending on your operating system (Windows, macOS, Linux). You would typically add export QUANDL_API_KEY='YOUR_QUANDL_API_KEY' to your shell's profile file (e.g., .bashrc, .zshrc, .profile) and then restart your terminal or source the file.


In [18]:
os.environ.get('QUANDL_API_KEY')

In [26]:
import quandl
import pandas as pd

quandl.ApiConfig.api_key = os.environ.get('QUANDL_API_KEY')

# EOD/META = Meta (Facebook) from EOD dataset (paid)
try:
    data = quandl.get('WIKI/TSLA', start_date='2019-01-01')
    print(data.head())
except Exception as e:
    print("Error:", e)


Error: (Status 403) Something went wrong. Please try again. If you continue to have problems, please contact us at connect@quandl.com.


### FRED

In [27]:
start = datetime(2010, 1, 1)

end = datetime(2013, 1, 27)

gdp = web.DataReader('GDP', 'fred', start, end)

gdp.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 13 entries, 2010-01-01 to 2013-01-01
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   GDP     13 non-null     float64
dtypes: float64(1)
memory usage: 208.0 bytes


In [28]:
gdp.head()

,GDP
DATE,
2010-01-01,14764.610
2010-04-01,14980.193
2010-07-01,15141.607
2010-10-01,15309.474
2011-01-01,15351.448


In [29]:
inflation = web.DataReader(['CPIAUCSL', 'CPILFESL'], 'fred', start, end)
inflation.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 37 entries, 2010-01-01 to 2013-01-01
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   CPIAUCSL  37 non-null     float64
 1   CPILFESL  37 non-null     float64
dtypes: float64(2)
memory usage: 888.0 bytes


### Fama/French

In [30]:
from pandas_datareader.famafrench import get_available_datasets
get_available_datasets()

['F-F_Research_Data_Factors',
 'F-F_Research_Data_Factors_weekly',
 'F-F_Research_Data_Factors_daily',
 'F-F_Research_Data_5_Factors_2x3',
 'F-F_Research_Data_5_Factors_2x3_daily',
 'Portfolios_Formed_on_ME',
 'Portfolios_Formed_on_ME_Wout_Div',
 'Portfolios_Formed_on_ME_Daily',
 'Portfolios_Formed_on_BE-ME',
 'Portfolios_Formed_on_BE-ME_Wout_Div',
 'Portfolios_Formed_on_BE-ME_Daily',
 'Portfolios_Formed_on_OP',
 'Portfolios_Formed_on_OP_Wout_Div',
 'Portfolios_Formed_on_OP_Daily',
 'Portfolios_Formed_on_INV',
 'Portfolios_Formed_on_INV_Wout_Div',
 'Portfolios_Formed_on_INV_Daily',
 '6_Portfolios_2x3',
 '6_Portfolios_2x3_Wout_Div',
 '6_Portfolios_2x3_weekly',
 '6_Portfolios_2x3_daily',
 '25_Portfolios_5x5',
 '25_Portfolios_5x5_Wout_Div',
 '25_Portfolios_5x5_Daily',
 '100_Portfolios_10x10',
 '100_Portfolios_10x10_Wout_Div',
 '100_Portfolios_10x10_Daily',
 '6_Portfolios_ME_OP_2x3',
 '6_Portfolios_ME_OP_2x3_Wout_Div',
 '6_Portfolios_ME_OP_2x3_daily',
 '25_Portfolios_ME_OP_5x5',
 '25_Portf

In [31]:
ds = web.DataReader('5_Industry_Portfolios', 'famafrench')
print(ds['DESCR'])

5 Industry Portfolios
---------------------

This file was created using the 202504 CRSP database. It contains value- and equal-weighted returns for 5 industry portfolios. The portfolios are constructed at the end of June. The annual returns are from January to December. Missing data are indicated by -99.99 or -999. Copyright 2025 Eugene F. Fama and Kenneth R. French

  0 : Average Value Weighted Returns -- Monthly (59 rows x 5 cols)
  1 : Average Equal Weighted Returns -- Monthly (59 rows x 5 cols)
  2 : Average Value Weighted Returns -- Annual (5 rows x 5 cols)
  3 : Average Equal Weighted Returns -- Annual (5 rows x 5 cols)
  4 : Number of Firms in Portfolios (59 rows x 5 cols)
  5 : Average Firm Size (59 rows x 5 cols)
  6 : Sum of BE / Sum of ME (5 rows x 5 cols)
  7 : Value-Weighted Average of BE/ME (5 rows x 5 cols)


/var/folders/_1/hlx9skzx2wd47fbwgn286hkm0000gn/T/ipykernel_2169/3709057983.py:1: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ds = web.DataReader('5_Industry_Portfolios', 'famafrench')
/var/folders/_1/hlx9skzx2wd47fbwgn286hkm0000gn/T/ipykernel_2169/3709057983.py:1: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ds = web.DataReader('5_Industry_Portfolios', 'famafrench')
/var/folders/_1/hlx9skzx2wd47fbwgn286hkm0000gn/T/ipykernel_2169/3709057983.py:1: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ds = web.DataReader('5_Industry_Port

### World Bank

In [32]:
from pandas_datareader import wb
gdp_variables = wb.search('gdp.*capita.*const')
gdp_variables.head()

,id,name,unit,source,sourceNote,sourceOrganization,topics
691,6.0.GDPpc_constant,"GDP per capita, PPP (constant 2011 internation...",,LAC Equity Lab,GDP per capita based on purchasing power parit...,b'World Development Indicators (World Bank)',Economy & Growth
11203,NY.GDP.PCAP.KD,GDP per capita (constant 2015 US$),,World Development Indicators,GDP per capita is gross domestic product divid...,"b'World Bank national accounts data, and OECD ...",Economy & Growth
11205,NY.GDP.PCAP.KN,GDP per capita (constant LCU),,World Development Indicators,GDP per capita is gross domestic product divid...,"b'World Bank national accounts data, and OECD ...",Economy & Growth
11207,NY.GDP.PCAP.PP.KD,"GDP per capita, PPP (constant 2021 internation...",,World Development Indicators,GDP per capita based on purchasing power parit...,"b'International Comparison Program, World Bank...",Economy & Growth
11208,NY.GDP.PCAP.PP.KD.87,"GDP per capita, PPP (constant 1987 internation...",,WDI Database Archives,,b'',


In [33]:
wb_data = wb.download(indicator='NY.GDP.PCAP.KD', 
                      country=['US', 'CA', 'MX'], 
                      start=1990, 
                      end=2019)
wb_data.head()

/var/folders/_1/hlx9skzx2wd47fbwgn286hkm0000gn/T/ipykernel_2169/873529061.py:1: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  wb_data = wb.download(indicator='NY.GDP.PCAP.KD',


NY.GDP.PCAP.KD
country year                
Canada  2019    45100.291490
        2018    44907.343684
        2017    44339.388669
        2016    43551.342602
        2015    43594.194105

### OECD

In [43]:
import pandasdmx as sdmx
sdmx.list_sources()


['ABS',
 'ABS_XML',
 'BBK',
 'BIS',
 'CD2030',
 'ECB',
 'EC_COMP',
 'EC_EMPL',
 'EC_GROW',
 'ESTAT',
 'ILO',
 'IMF',
 'INEGI',
 'INSEE',
 'ISTAT',
 'LSD',
 'NB',
 'NBB',
 'OECD',
 'SGR',
 'SPC',
 'STAT_EE',
 'UNICEF',
 'UNSD',
 'WB',
 'WB_WDI']

In [48]:
# 1. Define the agency ID for OECD
# 'OECD' is the commonly used ID for the OECD SDMX API endpoint
agency_id = 'OECD'

# 2. Define the dataset ID for Main Economic Indicators (MEI)
# The dataset ID for Main Economic Indicators is usually 'MEI_CLI' or similar.
# You might need to check OECD's SDMX API documentation for the exact ID if this doesn't work.
# A common one for composite leading indicators (part of MEI) is 'MEI_CLI'
dataset_id = 'MEI_CLI' # Or 'MEI_ARCHIVE', 'MEI_FIN' depending on specific MEI data needed

try:
    # 3. Get the data from the SDMX API
    # The sdmx.read_sdmx() function handles the connection and data retrieval
    # You can also add parameters like key (for dimensions), start and end dates
    print(f"Attempting to retrieve dataset: {dataset_id} from {agency_id}...")
    # data_response = sdmx.Request(f"https://stats.oecd.org/sdmx-json/data/{dataset_id}")
    data_response = sdmx.Request(agency_id).get(resource_id=dataset_id)


    # For a more specific query, you might add filters, e.g.:
    # data_response = sdmx.read_sdmx(f"https://stats.oecd.org/sdmx-json/data/{dataset_id}/all/OECD.AUS+USA.M.IXOB.GP.IX.M?startPeriod=2010&endPeriod=2020")
    # The parameters depend heavily on the specific dataset's dimensions.

    # 4. Convert the dataset to a pandas DataFrame
    # The .to_pandas() method provides a convenient way to get the data into a DataFrame
    df = data_response.data.to_pandas()

    print("\nSuccessfully retrieved MEI dataset sample:")
    print(df.head())
    print(f"\nDataset shape: {df.shape}")

except Exception as e:
    print(f"\nAn error occurred while retrieving the data: {e}")
    print("Please check the dataset ID, agency ID, and your internet connection.")
    print("You might need to consult the OECD SDMX API documentation for specific dataset codes and query parameters.")

Attempting to retrieve dataset: MEI_CLI from OECD...

An error occurred while retrieving the data: None
Please check the dataset ID, agency ID, and your internet connection.
You might need to consult the OECD SDMX API documentation for specific dataset codes and query parameters.


### EuroStat

In [50]:
import eurostat

# Get the working table of contents to explore dataset codes
toc = eurostat.get_toc_df()
print(toc.head())

# Now fetch the "Rail accidents by type of accident" dataset
df = eurostat.get_data_df('tran_sf_railac')
print(df.head())


                                               title         code     type  \
0  Distribution of digital platform workers (at l...  LFST_DPW_07  dataset   
1  Distribution of digital platform workers (at l...  LFST_DPW_08  dataset   
2  Distribution of digital platform workers (at l...  LFST_DPW_09  dataset   
3  Distribution of digital platform workers (at l...  LFST_DPW_10  dataset   
4  Distribution of digital platform workers (at l...  LFST_DPW_11  dataset   

        last update of data last table structure change data start data end  
0  2024-07-17T23:00:00+0200    2024-07-17T23:00:00+0200       2022     2022  
1  2024-07-17T23:00:00+0200    2024-07-17T23:00:00+0200       2022     2022  
2  2024-07-17T23:00:00+0200    2024-07-17T23:00:00+0200       2022     2022  
3  2024-07-17T23:00:00+0200    2024-07-17T23:00:00+0200       2022     2022  
4  2024-07-17T23:00:00+0200    2024-07-17T23:00:00+0200       2022     2022  
  freq unit accident geo\TIME_PERIOD  2006  2007  2008  2009  2

In [51]:
# List the dataset parameters
params = eurostat.get_pars('tran_sf_railac')
print(params)

# For example, check available values for 'geo' (countries)
geos = eurostat.get_par_values('tran_sf_railac', 'geo')
print(geos[:10])


['freq', 'unit', 'accident', 'geo']
['EU27_2020', 'BE', 'BG', 'CZ', 'DK', 'DE', 'EE', 'IE', 'EL', 'ES']




### Stooq

Google finance stopped providing common index data download. The Stooq site had this data for download for a while but is currently broken, awaiting release of [fix](https://github.com/pydata/pandas-datareader/issues/594)

In [13]:
index_url = 'https://stooq.com/t/'
ix = pd.read_html(index_url)
len(ix)

46

In [14]:
f = web.DataReader('^SPX', 'stooq', start='20000101')
f.info()

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Empty DataFrame

In [15]:
f.head()

""
No data


### NASDAQ Symbols

In [23]:
from pandas_datareader.nasdaq_trader import get_nasdaq_symbols
symbols = get_nasdaq_symbols()
symbols.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8701 entries, A to ZYXI
Data columns (total 11 columns):
Nasdaq Traded       8701 non-null bool
Security Name       8701 non-null object
Listing Exchange    8701 non-null category
Market Category     8701 non-null object
ETF                 8701 non-null bool
Round Lot Size      8701 non-null float64
Test Issue          8701 non-null bool
Financial Status    3411 non-null category
CQS Symbol          5290 non-null object
NASDAQ Symbol       8701 non-null object
NextShares          8701 non-null bool
dtypes: bool(4), category(2), float64(1), object(4)
memory usage: 459.2+ KB


In [24]:
url = 'https://www.nasdaq.com/screening/companies-by-industry.aspx?exchange=NASDAQ'
res = pd.read_html(url)
len(res)

4

In [25]:
for r in res:
    print(r.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 2 columns):
0    1 non-null object
1    1 non-null object
dtypes: object(2)
memory usage: 96.0+ bytes
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 6 columns):
Name          101 non-null object
Symbol        51 non-null object
Market Cap    47 non-null object
Country       51 non-null object
IPO Year      28 non-null object
Subsector     51 non-null object
dtypes: object(6)
memory usage: 4.8+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 1 columns):
0    1 non-null object
dtypes: object(1)
memory usage: 88.0+ bytes
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 1 columns):
0    1 non-null object
dtypes: object(1)
memory usage: 88.0+ bytes
None


### Tiingo

Requires [signing up](https://api.tiingo.com/) and storing API key in environment

In [26]:
df = web.get_data_tiingo('GOOG', api_key=os.getenv('TIINGO_API_KEY'))

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 1244 entries, (GOOG, 2014-03-27 00:00:00) to (GOOG, 2019-03-06 00:00:00)
Data columns (total 12 columns):
adjClose       1244 non-null float64
adjHigh        1244 non-null float64
adjLow         1244 non-null float64
adjOpen        1244 non-null float64
adjVolume      1244 non-null int64
close          1244 non-null float64
divCash        1244 non-null float64
high           1244 non-null float64
low            1244 non-null float64
open           1244 non-null float64
splitFactor    1244 non-null float64
volume         1244 non-null int64
dtypes: float64(10), int64(2)
memory usage: 130.1+ KB
